# Feature Engineering — Dataset Final

**Objetivo:** Crear el dataset definitivo para modelado:

1. Cargar las 3 tablas raw (application, bureau, previous_application)
2. Creamos las features definitivas desde las tablas secundarias
3. Merge central
4. Features derivadas de negocio (ratios, interacciones)
5. Guardamos el dataset final para preprocessing


---
## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

---
## 2. Carga de datos raw

In [2]:
app = pd.read_csv('../data/raw/application_train.csv')
bureau = pd.read_csv('../data/raw/bureau.csv')
prev = pd.read_csv('../data/raw/previous_application.csv')

print(f'application_train: {app.shape[0]:,} filas x {app.shape[1]} columnas')
print(f'bureau:            {bureau.shape[0]:,} filas x {bureau.shape[1]} columnas')
print(f'previous_application: {prev.shape[0]:,} filas x {prev.shape[1]} columnas')

application_train: 307,511 filas x 122 columnas
bureau:            1,716,428 filas x 17 columnas
previous_application: 1,670,214 filas x 37 columnas


---
## 3. Features desde bureau

El historial crediticio externo indica si el cliente es propenso a defaultear.

In [3]:
# === AGREGACIONES NUMÉRICAS ===
bureau_num = bureau.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU': 'count',
    'AMT_CREDIT_SUM': ['sum', 'mean', 'max'],
    'AMT_CREDIT_SUM_DEBT': ['sum', 'mean'],
    'AMT_CREDIT_SUM_OVERDUE': 'sum',
    'AMT_CREDIT_MAX_OVERDUE': 'max',
    'AMT_ANNUITY': ['mean', 'max'],
    'DAYS_CREDIT': ['mean', 'min', 'max'],
    'CREDIT_DAY_OVERDUE': ['mean', 'max'],
    'DAYS_CREDIT_UPDATE': 'mean',
    'CNT_CREDIT_PROLONG': 'sum',
})

bureau_num.columns = ['_'.join(col).strip() for col in bureau_num.columns]
bureau_num = bureau_num.reset_index()

bureau_num = bureau_num.rename(columns={
    'SK_ID_BUREAU_count': 'bureau_total_credits',
    'AMT_CREDIT_SUM_sum': 'bureau_total_credit_sum',
    'AMT_CREDIT_SUM_mean': 'bureau_avg_credit',
    'AMT_CREDIT_SUM_max': 'bureau_max_credit',
    'AMT_CREDIT_SUM_DEBT_sum': 'bureau_total_debt',
    'AMT_CREDIT_SUM_DEBT_mean': 'bureau_avg_debt',
    'AMT_CREDIT_SUM_OVERDUE_sum': 'bureau_total_overdue',
    'AMT_CREDIT_MAX_OVERDUE_max': 'bureau_max_overdue',
    'DAYS_CREDIT_mean': 'bureau_avg_days_credit',
    'DAYS_CREDIT_min': 'bureau_oldest_credit_days',
    'DAYS_CREDIT_max': 'bureau_newest_credit_days',
    'CREDIT_DAY_OVERDUE_mean': 'bureau_avg_overdue_days',
    'CREDIT_DAY_OVERDUE_max': 'bureau_max_overdue_days',
})

print(f'Features numéricas de bureau: {bureau_num.shape[1] - 1}')
bureau_num.head()

Features numéricas de bureau: 17


,SK_ID_CURR,bureau_total_credits,bureau_total_credit_sum,bureau_avg_credit,bureau_max_credit,bureau_total_debt,bureau_avg_debt,bureau_total_overdue,bureau_max_overdue,AMT_ANNUITY_mean,AMT_ANNUITY_max,bureau_avg_days_credit,bureau_oldest_credit_days,bureau_newest_credit_days,bureau_avg_overdue_days,bureau_max_overdue_days,DAYS_CREDIT_UPDATE_mean,CNT_CREDIT_PROLONG_sum
0,100001,7,1453365.0000,207623.5714,378000.0000,596686.5000,85240.9286,0.0000,NaN,3545.3571,10822.5000,-735.0000,-1572,-49,0.0000,0,-93.1429,0
1,100002,8,865055.5650,108131.9456,450000.0000,245781.0000,49156.2000,0.0000,5043.6450,0.0000,0.0000,-874.0000,-1437,-103,0.0000,0,-499.8750,0
2,100003,4,1017400.5000,254350.1250,810000.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,-1400.7500,-2586,-606,0.0000,0,-816.0000,0
3,100004,2,189037.8000,94518.9000,94537.8000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,-867.0000,-1326,-408,0.0000,0,-532.0000,0
4,100005,3,657126.0000,219042.0000,568800.0000,568408.5000,189469.5000,0.0000,0.0000,1420.5000,4261.5000,-190.6667,-373,-62,0.0000,0,-54.3333,0


In [4]:
# === PROPORCIONES DE ESTADO ===
bureau_status = pd.get_dummies(bureau[['SK_ID_CURR', 'CREDIT_ACTIVE']])
bureau_status = bureau_status.groupby('SK_ID_CURR').sum().reset_index()

# Calcular proporciones
status_cols = [c for c in bureau_status.columns if c.startswith('CREDIT_ACTIVE_')]
total = bureau_status[status_cols].sum(axis=1)
for col in status_cols:
    bureau_status[f'{col}_pct'] = bureau_status[col] / total

print(f'Proporciones de estado: {len(status_cols)} categorías')

Proporciones de estado: 4 categorías


In [5]:
# === FEATURES DERIVADAS DE NEGOCIO ===
bureau_derived = pd.DataFrame()
bureau_derived['SK_ID_CURR'] = bureau['SK_ID_CURR'].unique()

# Conteo de estados por cliente
refused = bureau.groupby('SK_ID_CURR')['CREDIT_ACTIVE'].apply(lambda x: (x == 'Bad debt').sum()).reset_index(name='bureau_bad_debt_count')
active = bureau.groupby('SK_ID_CURR')['CREDIT_ACTIVE'].apply(lambda x: (x == 'Active').sum()).reset_index(name='bureau_active_count')

bureau_derived = bureau_derived.merge(refused, on='SK_ID_CURR', how='left')
bureau_derived = bureau_derived.merge(active, on='SK_ID_CURR', how='left')
bureau_derived = bureau_derived.merge(bureau_num[['SK_ID_CURR', 'bureau_total_credits']], on='SK_ID_CURR', how='left')

# Ratios
bureau_derived['bureau_bad_debt_rate'] = bureau_derived['bureau_bad_debt_count'] / bureau_derived['bureau_total_credits']
bureau_derived['bureau_active_rate'] = bureau_derived['bureau_active_count'] / bureau_derived['bureau_total_credits']

print('Features derivadas de bureau:')
bureau_derived.head()

Features derivadas de bureau:


,SK_ID_CURR,bureau_bad_debt_count,bureau_active_count,bureau_total_credits,bureau_bad_debt_rate,bureau_active_rate
0,215354,0,6,11,0.0000,0.5455
1,162297,0,3,6,0.0000,0.5000
2,402440,0,1,1,0.0000,1.0000
3,238881,0,3,8,0.0000,0.3750
4,222183,0,5,8,0.0000,0.6250


In [6]:
# === UNIR TODO BUREAU ===
bureau_features = bureau_num.merge(bureau_status[['SK_ID_CURR'] + [f'{c}_pct' for c in status_cols]], 
                                on='SK_ID_CURR', how='left')
bureau_features = bureau_features.merge(bureau_derived[['SK_ID_CURR', 'bureau_bad_debt_rate', 'bureau_active_rate']], 
                                        on='SK_ID_CURR', how='left')

# Ratios de negocio
bureau_features['bureau_debt_ratio'] = bureau_features['bureau_total_debt'] / bureau_features['bureau_total_credit_sum']
bureau_features['bureau_overdue_ratio'] = bureau_features['bureau_total_overdue'] / bureau_features['bureau_total_credit_sum']

print(f'Features de bureau listas: {bureau_features.shape[1] - 1} columnas')

Features de bureau listas: 25 columnas


---
## 4. Features desde previous_application

Justificación de negocio: El comportamiento previo del cliente dentro de Home Credit predice su comportamiento futuro.

In [7]:
# === AGREGACIONES NUMÉRICAS ===
prev_num = prev.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'AMT_APPLICATION': ['sum', 'mean', 'max'],
    'AMT_CREDIT': ['sum', 'mean', 'max'],
    'AMT_ANNUITY': ['mean', 'max'],
    'AMT_DOWN_PAYMENT': ['sum', 'mean'],
    'DAYS_DECISION': ['mean', 'min', 'max'],
    'CNT_PAYMENT': ['mean', 'max'],
    'RATE_DOWN_PAYMENT': 'mean',
})

prev_num.columns = ['_'.join(col).strip() for col in prev_num.columns]
prev_num = prev_num.reset_index()

prev_num = prev_num.rename(columns={
    'SK_ID_PREV_count': 'prev_total_applications',
    'AMT_APPLICATION_sum': 'prev_total_requested',
    'AMT_APPLICATION_mean': 'prev_avg_requested',
    'AMT_APPLICATION_max': 'prev_max_requested',
    'AMT_CREDIT_sum': 'prev_total_approved',
    'AMT_CREDIT_mean': 'prev_avg_approved',
    'DAYS_DECISION_mean': 'prev_avg_decision_days',
    'DAYS_DECISION_min': 'prev_oldest_decision_days',
    'DAYS_DECISION_max': 'prev_newest_decision_days',
})

print(f'Features numéricas de previous: {prev_num.shape[1] - 1}')
prev_num.head()

Features numéricas de previous: 17


,SK_ID_CURR,prev_total_applications,prev_total_requested,prev_avg_requested,prev_max_requested,prev_total_approved,prev_avg_approved,AMT_CREDIT_max,AMT_ANNUITY_mean,AMT_ANNUITY_max,AMT_DOWN_PAYMENT_sum,AMT_DOWN_PAYMENT_mean,prev_avg_decision_days,prev_oldest_decision_days,prev_newest_decision_days,CNT_PAYMENT_mean,CNT_PAYMENT_max,RATE_DOWN_PAYMENT_mean
0,100001,1,24835.5000,24835.5000,24835.5000,23787.0000,23787.0000,23787.0000,3951.0000,3951.0000,2520.0000,2520.0000,-1740.0000,-1740,-1740,8.0000,8.0000,0.1043
1,100002,1,179055.0000,179055.0000,179055.0000,179055.0000,179055.0000,179055.0000,9251.7750,9251.7750,0.0000,0.0000,-606.0000,-606,-606,24.0000,24.0000,0.0000
2,100003,3,1306309.5000,435436.5000,900000.0000,1452573.0000,484191.0000,1035882.0000,56553.9900,98356.9950,6885.0000,3442.5000,-1305.0000,-2341,-746,10.0000,12.0000,0.0500
3,100004,1,24282.0000,24282.0000,24282.0000,20106.0000,20106.0000,20106.0000,5357.2500,5357.2500,4860.0000,4860.0000,-815.0000,-815,-815,4.0000,4.0000,0.2120
4,100005,2,44617.5000,22308.7500,44617.5000,40153.5000,20076.7500,40153.5000,4813.2000,4813.2000,4464.0000,4464.0000,-536.0000,-757,-315,12.0000,12.0000,0.1090


In [8]:
# === PROPORCIONES DE ESTADO ===
prev_status = pd.get_dummies(prev[['SK_ID_CURR', 'NAME_CONTRACT_STATUS']])
prev_status = prev_status.groupby('SK_ID_CURR').sum().reset_index()

status_cols_prev = [c for c in prev_status.columns if c.startswith('NAME_CONTRACT_STATUS_')]
total_prev = prev_status[status_cols_prev].sum(axis=1)
for col in status_cols_prev:
    prev_status[f'{col}_pct'] = prev_status[col] / total_prev

print(f'Proporciones de estado: {len(status_cols_prev)} categorías')

Proporciones de estado: 4 categorías


In [9]:
# === FEATURES DERIVADAS DE NEGOCIO ===
prev_derived = pd.DataFrame()
prev_derived['SK_ID_CURR'] = prev['SK_ID_CURR'].unique()

refused_prev = prev.groupby('SK_ID_CURR')['NAME_CONTRACT_STATUS'].apply(lambda x: (x == 'Refused').sum()).reset_index(name='prev_refused_count')
approved_prev = prev.groupby('SK_ID_CURR')['NAME_CONTRACT_STATUS'].apply(lambda x: (x == 'Approved').sum()).reset_index(name='prev_approved_count')

prev_derived = prev_derived.merge(refused_prev, on='SK_ID_CURR', how='left')
prev_derived = prev_derived.merge(approved_prev, on='SK_ID_CURR', how='left')
prev_derived = prev_derived.merge(prev_num[['SK_ID_CURR', 'prev_total_applications']], on='SK_ID_CURR', how='left')

prev_derived['prev_reject_rate'] = prev_derived['prev_refused_count'] / prev_derived['prev_total_applications']
prev_derived['prev_approve_rate'] = prev_derived['prev_approved_count'] / prev_derived['prev_total_applications']

print('Features derivadas de previous_application:')
prev_derived.head()

Features derivadas de previous_application:


,SK_ID_CURR,prev_refused_count,prev_approved_count,prev_total_applications,prev_reject_rate,prev_approve_rate
0,271877,1,2,3,0.3333,0.6667
1,108129,0,6,6,0.0000,1.0000
2,122040,0,3,4,0.0000,0.7500
3,176158,15,6,23,0.6522,0.2609
4,202054,13,8,25,0.5200,0.3200


In [10]:
# === UNIR TODO PREVIOUS ===
prev_features = prev_num.merge(prev_status[['SK_ID_CURR'] + [f'{c}_pct' for c in status_cols_prev]], 
                            on='SK_ID_CURR', how='left')
prev_features = prev_features.merge(prev_derived[['SK_ID_CURR', 'prev_reject_rate', 'prev_approve_rate']], 
                                    on='SK_ID_CURR', how='left')

# Ratio de negocio: cuánto le dieron vs cuánto pidió
prev_features['prev_credit_ratio'] = prev_features['prev_total_approved'] / prev_features['prev_total_requested']

print(f'Features de previous listas: {prev_features.shape[1] - 1} columnas')

Features de previous listas: 24 columnas


---
## 5. Merge central

Unimos todo a `application_train`.

In [11]:
TARGET_COL = 'TARGET'
df = app.copy()

print(f'Start: {df.shape[1]} columnas')

# Merge bureau
df = df.merge(bureau_features, on='SK_ID_CURR', how='left')
print(f'Después de bureau: {df.shape[1]} columnas (+{df.shape[1] - app.shape[1]})')

# Merge previous_application
df = df.merge(prev_features, on='SK_ID_CURR', how='left')
print(f'Después de previous: {df.shape[1]} columnas (+{df.shape[1] - app.shape[1]})')

print(f'\nDataset final: {df.shape[0]:,} filas x {df.shape[1]} columnas')

Start: 122 columnas
Después de bureau: 147 columnas (+25)
Después de previous: 171 columnas (+49)

Dataset final: 307,511 filas x 171 columnas


---
## 6. Features derivadas de negocio

Creaciones basadas en lógica de negocio que combinan information de múltiples tablas.

In [12]:
# === RATIOS DE NEGOCIO ===

# Cuánto del crédito solicitado le aprobaron
df['credit_approval_ratio'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE'].replace(0, np.nan)

# Cuánto paga mensualmente vs su ingreso
df['annuity_to_income'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL'].replace(0, np.nan)

# Cuánto debe vs cuánto gana
df['credit_to_income'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL'].replace(0, np.nan)

# Ratio de bienes al crédito
df['goods_to_credit'] = df['AMT_GOODS_PRICE'] / df['AMT_CREDIT'].replace(0, np.nan)

In [13]:
# === FEATURES DE TIEMPO ===

# Edad en años (DAYS_BIRTH es negativo)
df['age_years'] = (-df['DAYS_BIRTH'] / 365.25).astype(float)

# Antigüedad laboral en años
df['employment_years'] = (-df['DAYS_EMPLOYED'] / 365.25).astype(float)

# Ratio antigüedad laboral / edad
df['employment_ratio'] = df['employment_years'] / df['age_years'].replace(0, np.nan)

# Días desde última modificación de registro
df['registration_years'] = (-df['DAYS_REGISTRATION'] / 365.25).astype(float)

print('Features de tiempo creadas.')

Features de tiempo creadas.


In [14]:
# === FEATURES DE EXTERNAL SOURCE ===
# Promedio de las 3 fuentes externas
ext_cols = [c for c in df.columns if c.startswith('EXT_SOURCE_')]
if ext_cols:
    df['ext_source_mean'] = df[ext_cols].mean(axis=1)
    df['ext_source_std'] = df[ext_cols].std(axis=1)
    df['ext_source_min'] = df[ext_cols].min(axis=1)
    df['ext_source_max'] = df[ext_cols].max(axis=1)
    print(f'Features de external source creadas: 4 (usando {ext_cols})')
else:
    print('No se encontraron columnas EXT_SOURCE_')

Features de external source creadas: 4 (usando ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'])


In [15]:
# === FEATURES DE DOCUMENTOS ===
doc_cols = [c for c in df.columns if c.startswith('FLAG_DOCUMENT_')]
df['total_documents'] = df[doc_cols].sum(axis=1)
print(f'Features de documentos creadas: 1 (total de {len(doc_cols)} documentos)')

Features de documentos creadas: 1 (total de 20 documentos)


In [16]:
# === FEATURES DE CONTACTO ===
contact_cols = ['FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL']
contact_cols = [c for c in contact_cols if c in df.columns]
df['total_contact_methods'] = df[contact_cols].sum(axis=1)
print(f'Features de contacto creadas: 1 (total de {len(contact_cols)} métodos)')

Features de contacto creadas: 1 (total de 5 métodos)


---
## 7. Resumen del dataset final

In [17]:
# Separar features por origen
app_cols = [c for c in app.columns]
bureau_only = [c for c in df.columns if c.startswith('bureau_')]
prev_only = [c for c in df.columns if c.startswith('prev_')]
engineered = [c for c in df.columns if c not in app_cols and c not in bureau_only and c not in prev_only]

print('RESUMEN DEL DATASET FINAL')
print('=' * 50)
print(f'Total columnas:   {df.shape[1]}')
print(f'  Originales:     {len(app_cols)}')
print(f'  De bureau:       {len(bureau_only)}')
print(f'  De previous:     {len(prev_only)}')
print(f'  Ingeniería:      {len(engineered)}')
print()
print(f'Total filas:      {df.shape[0]:,}')
print(f'Target:           {TARGET_COL}')
print(f'Ratio target:     {df[TARGET_COL].value_counts().to_dict()}')

RESUMEN DEL DATASET FINAL
Total columnas:   185
  Originales:     122
  De bureau:       17
  De previous:     12
  Ingeniería:      34

Total filas:      307,511
Target:           TARGET
Ratio target:     {0: 282686, 1: 24825}


In [19]:
# Features de ingeniería creadas
print('FEATURES DE INGENIERÍA:')
for f in sorted(engineered):
    print(f'  - {f}')

FEATURES DE INGENIERÍA:
  - AMT_ANNUITY_max_x
  - AMT_ANNUITY_max_y
  - AMT_ANNUITY_mean_x
  - AMT_ANNUITY_mean_y
  - AMT_CREDIT_max
  - AMT_DOWN_PAYMENT_mean
  - AMT_DOWN_PAYMENT_sum
  - CNT_CREDIT_PROLONG_sum
  - CNT_PAYMENT_max
  - CNT_PAYMENT_mean
  - CREDIT_ACTIVE_Active_pct
  - CREDIT_ACTIVE_Bad debt_pct
  - CREDIT_ACTIVE_Closed_pct
  - CREDIT_ACTIVE_Sold_pct
  - DAYS_CREDIT_UPDATE_mean
  - NAME_CONTRACT_STATUS_Approved_pct
  - NAME_CONTRACT_STATUS_Canceled_pct
  - NAME_CONTRACT_STATUS_Refused_pct
  - NAME_CONTRACT_STATUS_Unused offer_pct
  - RATE_DOWN_PAYMENT_mean
  - age_years
  - annuity_to_income
  - credit_approval_ratio
  - credit_to_income
  - employment_ratio
  - employment_years
  - ext_source_max
  - ext_source_mean
  - ext_source_min
  - ext_source_std
  - goods_to_credit
  - registration_years
  - total_contact_methods
  - total_documents


In [20]:
# Nulos en el dataset final
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)

null_report = pd.DataFrame({'nulos': nulls, 'pct': nulls_pct}).query('nulos > 0').sort_values('pct', ascending=False)
print(f'Columnas con nulos: {len(null_report)} de {df.shape[1]}')
print(f'\nTop 15:')
null_report.head(15)

Columnas con nulos: 123 de 185

Top 15:


,nulos,pct
AMT_ANNUITY_max_x,227502,73.9800
AMT_ANNUITY_mean_x,227502,73.9800
COMMONAREA_AVG,214865,69.8700
COMMONAREA_MEDI,214865,69.8700
COMMONAREA_MODE,214865,69.8700
NONLIVINGAPARTMENTS_MEDI,213514,69.4300
NONLIVINGAPARTMENTS_MODE,213514,69.4300
NONLIVINGAPARTMENTS_AVG,213514,69.4300
FONDKAPREMONT_MODE,210295,68.3900
LIVINGAPARTMENTS_MEDI,210199,68.3500


---
## 8. Guardar dataset final

In [21]:
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/application_train_features.csv', index=False)
print(f'Guardado: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Ruta: data/processed/application_train_features.csv')

Guardado: 307,511 filas x 185 columnas
Ruta: data/processed/application_train_features.csv
